In [2]:
import numpy as np
import torch
import torch.nn as nn
import joblib
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

pio.renderers.default = 'vscode'
pio.templates.default = 'plotly_white'

class Pt100Net(nn.Module):
    def __init__(self, hidden_sizes=(64, 64, 32), dropout=0.1):
        super().__init__()
        layers = []
        prev = 1
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.Tanh(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

ckpt = torch.load('/home/arseniy/Document/Unic/Program/Прочее/Plat_ml/notebooks/models/pt100_model.pth', map_location='cpu')
model = Pt100Net(ckpt['hidden_sizes'], ckpt['dropout'])
model.load_state_dict(ckpt['model_state'])
model.eval()

scaler_y = joblib.load('/home/arseniy/Document/Unic/Program/Прочее/Plat_ml/data/processed/scaler_y.pkl')
print(f"Модель загружена. Best val loss: {ckpt['best_val']:.6e}")
print(f"Эпох обучено: {ckpt['epochs_trained']}")

Модель загружена. Best val loss: 1.032697e-04
Эпох обучено: 360


In [4]:
test = np.load('/home/arseniy/Document/Unic/Program/Прочее/Plat_ml/data/processed/test.npz')
X_test, y_test = test['X'], test['y']

with torch.no_grad():
    X_t = torch.tensor(X_test, dtype=torch.float32)
    y_pred_s = model(X_t).numpy()

y_pred = scaler_y.inverse_transform(y_pred_s)
y_true = scaler_y.inverse_transform(y_test)

rmse = np.sqrt(np.mean((y_pred - y_true)**2))
mae  = np.mean(np.abs(y_pred - y_true))
max_err = np.max(np.abs(y_pred - y_true))

print("=== Метрики на TEST (Ом) ===")
print(f"RMSE:      {rmse:.6f}")
print(f"MAE:       {mae:.6f}")
print(f"Max |err|: {max_err:.6f}")

print("\n=== Метрики на TEST (°C, приближённо) ===")
print(f"RMSE:      {rmse/0.385:.6f}")
print(f"MAE:       {mae/0.385:.6f}")
print(f"Max |err|: {max_err/0.385:.6f}")

=== Метрики на TEST (Ом) ===
RMSE:      3.877250
MAE:       3.061377
Max |err|: 11.455902

=== Метрики на TEST (°C, приближённо) ===
RMSE:      10.070781
MAE:       7.951628
Max |err|: 29.755590


In [5]:
fig = go.Figure()

fig.add_trace(go.Scattergl(
    x=y_true.flatten(), y=y_pred.flatten(),
    mode='markers',
    marker=dict(size=4, opacity=0.5, color='steelblue'),
    name='Предсказания'
))

lims = [y_true.min(), y_true.max()]
fig.add_trace(go.Scatter(
    x=lims, y=lims, mode='lines',
    line=dict(dash='dash', color='red', width=2),
    name='Идеал (y=x)'
))

fig.update_xaxes(title_text='Истинное R, Ом')
fig.update_yaxes(title_text='Предсказанное R, Ом')
fig.update_layout(
    height=500,
    title_text='Test: предсказание vs истина',
    title_x=0.5
)
fig.show()